# Generation: reproduce every attempt

Read Chapter 29, Primer-PY and MG-2. Both languages load their prompts from one versioned resource. This executes 36 actual CPU continuations with the retained final checkpoint, then compares token IDs, bytes, stop causes and traces with the historical run. Timing and cross-stack identity are not promised. The historical file is not overwritten. Text: CC BY-SA 4.0. Code: Apache-2.0.

In [1]:
from pathlib import Path
import json
import sys
import importlib.util
ROOT = Path.cwd()
assert (ROOT / "src/config/book.mjs").is_file(), "Run from the book repository root"
def load_json(path):
    return json.loads((ROOT / path).read_text())
def module(name, path):
    spec = importlib.util.spec_from_file_location(name, ROOT / path)
    value = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(value)
    return value
runner = module("generation_matrix", "code/part-v/generation_matrix.py")
recorded = load_json("data/part-v/generation-run.json")
replayed = runner.run(write=False)
assert len(replayed["rows"]) == 36
for observed, retained in zip(replayed["rows"], recorded["rows"], strict=True):
    assert observed == retained
print("All 36 continuations, including their full probability traces, reproduced.")
print("Environment:", replayed["environment"])


All 36 continuations, including their full probability traces, reproduced.
Environment: {'python': '3.12.10', 'torch': '2.7.0', 'device': 'cpu', 'dtype': 'float64', 'threads': 1}


In [2]:
for locale in ["en", "zh-hans"]:
    rows = [r for r in replayed["rows"] if r["locale"] == locale]
    for row in rows:
        print(row["task"], row["policy"], row["seed"], repr(row["output"]["completion"]), row["output"]["stop"])
assert sum(r["strict_match"] for r in replayed["rows"]) == 0
assert all(r["output"]["valid_utf8"] for r in replayed["rows"])
print("Zero strict passes does not measure creative quality; inspect the complete continuation against the task-specific rubric.")


structured greedy 7 'red.' EOS
structured greedy 42 'red.' EOS
structured greedy 31415 'red.' EOS
structured sample-0.7-k5 7 't750on: n: old.' EOS
structured sample-0.7-k5 42 'red.' EOS
structured sample-0.7-k5 31415 'red.' EOS
structured sample-1.2-k5 7 't750ol: n: old.' EOS
structured sample-1.2-k5 42 'ped.' EOS
structured sample-1.2-k5 31415 'tNon: n: old.' EOS
creative greedy 7 'Answer: red.' EOS
creative greedy 42 'Answer: red.' EOS
creative greedy 31415 'Answer: red.' EOS
creative sample-0.7-k5 7 'Answer: red.' EOS
creative sample-0.7-k5 42 'Answer: red.' EOS
creative sample-0.7-k5 31415 'Answer: red.' EOS
creative sample-1.2-k5 7 'Answer: red.' EOS
creative sample-1.2-k5 42 'Answer: red.' EOS
creative sample-1.2-k5 31415 'Answer: red.' EOS
structured greedy 7 ' red red red t: red te. Ver: red t: red t' max_new_tokens
structured greedy 42 ' red red red t: red te. Ver: red t: red t' max_new_tokens
structured greedy 31415 ' red red red t: red te. Ver: red t: red t' max_new_tokens
s